In [90]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [91]:
import os
os.chdir(r'C:\Kaggle_Competition\Playground\S6E5-F1-Pit-Stops')
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import rankdata
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Externals
from src.utils import *
import config
import warnings
warnings.filterwarnings('ignore')

In [92]:
train = read_csv_file('data/raw/train.csv')
test = read_csv_file('data/raw/test.csv')

train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

train_ids = train[config.ID_COL].values
test_ids = test[config.ID_COL].values

print(f'Shape of train data: {train.shape}')
print(f'Shape of test data: {test.shape}')

# Data
X = train.drop(['id', config.TARGET], axis=1)
y = train[config.TARGET].values
X_test = test.drop('id', axis=1)

Shape of train data: (439140, 16)
Shape of test data: (188165, 15)


### Weighted Blend

In [ ]:
MODELS = [
    ("catgbm", r"artifacts\oof\CatGBM_v00_oof_proba.csv", r"artifacts\test_proba\CatGBM_v00_test_proba.csv"),
    ('lgbm', r'artifacts\oof\LGBM_ext_oof_proba.csv', r'artifacts\test_proba\LGBM_ext_test_proba.csv'),
    ('xgbm', r'artifacts\oof\XGB_v16_seed42_oof_proba.csv', r'artifacts\test_proba\XGB_v16_seed42_test_proba.csv'),
    ('histgbm', r'artifacts\oof\HISTGBM_v2_oof_proba.csv', r'artifacts\test_proba\HISTGBM_v2_test_proba.csv'),
    ('realmlp', r'artifacts\oof\REALMLP_v2_seed42_oof_proba.csv', r'artifacts\test_proba\REALMLP_v2_seed42_test_proba.csv'),
    ('autogluon', r'artifacts\oof\AutoGluon_v6_seed42_oof_proba.csv', r'artifacts\test_proba\AutoGluon_v6_seed42_test_proba.csv'),
    ('tabpfn', r'artifacts\oof\TAB-PFN_v2_seed42_oof_proba.csv', r'artifacts\test_proba\TAB-PFN_v2_seed42_test_proba.csv'),
    ('resnet', r'artifacts\oof\RESNET_v1_seed42_oof_proba.csv', r'artifacts\test_proba\RESNET_v1_seed42_test_proba.csv'),
    ('tabm', r'artifacts\oof\TABM_v1_seed42_oof_proba.csv', r'artifacts\test_proba\TABM_v1_seed42_test_proba.csv'),
    ('stack', r'artifacts\oof\STACK_v0_oof_proba.csv', r'artifacts\test_proba\STACK_v0_test_proba.csv'),
    ('stack_lr', r'artifacts\oof\stack_lr_seedavg_v1_oof_proba.csv', r'artifacts\test_proba\stack_lr_seedavg_v1_test_proba.csv'),
    #('stack_ridge', r'artifacts\oof\stack_ridge[0.954786]_oof_proba.csv', r'artifacts\test_proba\stack_ridge[0.954786]_test_proba.csv')
]

def load_pred(path: str, pred_col: str = "PitNextLap") -> np.ndarray:

    df = read_csv_file(path)

    if pred_col in df.columns:
        return df[pred_col].to_numpy(dtype=np.float32)

    return df.iloc[:, 1].to_numpy(dtype=np.float32)


# Numpy matrices
names = [m[0] for m in MODELS]

oof_mat = np.column_stack([
    load_pred(m[1]) for m in MODELS
]).astype(np.float32)

test_mat = np.column_stack([
    load_pred(m[2]) for m in MODELS
]).astype(np.float32)


single_scores = {
    name: roc_auc_score(y, oof_mat[:, i])
    for i, name in enumerate(names)
}

for name, score in single_scores.items():
    print(f"{name} alone: {score:.6f}")


# Objective function
def objective(trial):

    raw = np.array(
        [
            trial.suggest_float(f"w_{name}", -3.0, 3.0)
            for name in names
        ],
        dtype=np.float32
    )

    # softmax normalization
    weights = np.exp(raw - raw.max())
    weights /= weights.sum()

    preds = oof_mat @ weights

    return roc_auc_score(y, preds)

# Phase 1 -> Quasi Monte Carlo
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.QMCSampler(
        seed=config.SEED,
        qmc_type='sobol',
        scramble=True,
        independent_sampler=optuna.samplers.RandomSampler(seed=config.SEED),
        warn_asynchronous_seeding=True,
        warn_independent_sampling=True
    ),
)

study.optimize(
    objective,
    n_trials=1000,
    show_progress_bar=True,
    n_jobs=1 # no parallelization to preserve descrepancy
)

# Phase 2 -> Tree-Prazen Estimator
study.sampler = optuna.samplers.TPESampler(
    n_ei_candidates=64,
    seed=config.SEED,
    multivariate=False,
    warn_independent_sampling=True,
    n_startup_trials=0,
    constant_liar=True # inject fake constants
)

study.optimize(
    objective,
    n_trials=5000,
    show_progress_bar=True,
    n_jobs=-1
)

# Best weights
best_raw = np.array(
    [
        study.best_params[f"w_{name}"]
        for name in names
    ],
    dtype=np.float32
)

best_w = np.exp(best_raw - best_raw.max())
best_w /= best_w.sum()

# Final weighted blend
blend_oof = oof_mat @ best_w
blend_test = test_mat @ best_w

# Results
best_single = max(single_scores.values())

print(f"\n{'='*50}")

print(f"Best blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - best_single:.6f}")

print(f"\n{'='*50}")

for name, w in zip(names, best_w):
    print(f"{name} weight: {w:.4f} ({w:.1%})")

catgbm alone: 0.952983
lgbm alone: 0.952587
xgbm alone: 0.952570
histgbm alone: 0.945228
realmlp alone: 0.954065
autogluon alone: 0.951694
tabpfn alone: 0.949509
resnet alone: 0.942910
tabm alone: 0.946562
stack alone: 0.954471
stack_lr alone: 0.954110


  0%|          | 0/1000 [00:00<?, ?it/s]

In [23]:
# Saving files
RUN_NAME = "my_blend_[0.954899]_test_proba"
blend_oof_df = pd.DataFrame({"id": train_ids, "PitNextLap": blend_oof})
blend_test_df = pd.DataFrame({"id": test_ids, "PitNextLap": blend_test})

save_csv_file(
    blend_oof_df,
    config.OOF_PROBA_DIR / f"{RUN_NAME}_oof_proba.csv"
)

save_csv_file(
    blend_test_df,
    config.TEST_PROBA_DIR / f"{RUN_NAME}_test_proba.csv"
)

### Stacking

In [41]:
oof_files = sorted(config.OOF_PROBA_DIR.glob("*_oof_proba.csv"))
test_files = sorted(config.TEST_PROBA_DIR.glob("*_test_proba.csv"))

oof_df = pd.DataFrame({"id": train_ids})
test_df = pd.DataFrame({"id": test_ids})

for i, file in enumerate(oof_files, start=1):
    oof_df[f"model_{i}"] = read_csv_file(file).iloc[:, 1].values

for i, file in enumerate(test_files, start=1):
    test_df[f"model_{i}"] = read_csv_file(file).iloc[:, 1].values

FEATURES = [col for col in oof_df.columns if col != "id"]

X_meta = oof_df[FEATURES].values
X_meta_test = test_df[FEATURES].values
y_meta = np.asarray(y)

In [42]:
print(X_meta.shape)
print(X_meta_test.shape)
print(y_meta.shape)

(439140, 54)
(188165, 54)
(439140,)


In [43]:
def objective(trial):

    selected_cols = [
        col for col in FEATURES
        if trial.suggest_int(f"use_{col}", 0, 1)
    ]

    if len(selected_cols) == 0:
        return 0.5

    X_selected = oof_df[selected_cols].values

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = []

    for fold, (tr_idx, val_idx) in enumerate(
        skf.split(X_selected, y_meta)
    ):

        X_tr = X_selected[tr_idx]
        X_val = X_selected[val_idx]

        y_tr = y_meta[tr_idx]
        y_val = y_meta[val_idx]

        model = make_pipeline(
            StandardScaler(),
            RidgeClassifier(alpha=2.5, max_iter=6000)
        )

        model.fit(X_tr, y_tr)

        val_preds = model.decision_function(X_val)

        fold_auc = roc_auc_score(y_val, val_preds)

        scores.append(fold_auc)

        # Pruning
        trial.report(np.mean(scores), step=fold)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    study_name='MY_STACK_V2',
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=20)
)

study.optimize(
    objective,
    n_trials=4000,
    show_progress_bar=True,
    n_jobs=-1
)

  0%|          | 0/4000 [00:00<?, ?it/s]

In [44]:
# best models
best_cols = [
    col for col in FEATURES
    if study.best_params[f"use_{col}"] == 1
]

In [53]:
X_best = oof_df[best_cols].values
X_best_test = test_df[best_cols].values
y_meta = np.asarray(y)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=config.SEED
)

oof_preds = np.zeros(len(X_best))
test_preds = np.zeros(len(X_best_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_best, y_meta), start=1):

    X_tr, X_val = X_best[tr_idx], X_best[val_idx]
    y_tr, y_val = y_meta[tr_idx], y_meta[val_idx]

    model = make_pipeline(
        StandardScaler(),
        RidgeClassifier(alpha=2.5, max_iter=6000)
    )

    model.fit(X_tr, y_tr)

    val_preds = model.decision_function(X_val)
    tst_preds = model.decision_function(X_best_test)

    oof_preds[val_idx] = val_preds
    test_preds += tst_preds / config.N_FOLDS

    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"Fold {fold} AUC: {fold_auc:.6f}")

final_auc = roc_auc_score(y_meta, oof_preds)
print(f"\nOOF AUC: {final_auc:.6f}")

Fold 1 AUC: 0.955877
Fold 2 AUC: 0.953693
Fold 3 AUC: 0.954783
Fold 4 AUC: 0.954100
Fold 5 AUC: 0.955547

OOF AUC: 0.954786


In [50]:
oof_df_out = pd.DataFrame({"id": train_ids, "PitNextLap": oof_preds})
test_df_out = pd.DataFrame({"id": test_ids, "PitNextLap": test_preds})

save_csv_file(oof_df_out, config.OOF_PROBA_DIR / "stack_ridge[0.954786]_oof_proba.csv")
save_csv_file(test_df_out, config.TEST_PROBA_DIR / "stack_ridge[0.954786]_test_proba.csv")